
# ex1 - 信用卡詐欺偵測實驗

本 Notebook 包含：
- 監督式學習模型：Random Forest（含優化版本）
- 非監督式學習模型：KMeans（聚類 + 標籤對齊）
- 評估指標：Precision、Recall、F1-score、Classification Report


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, silhouette_score
)
import kagglehub


In [ ]:

def evaluation(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{model_name} Evaluation:')
    print('===' * 15)
    print('         Accuracy:', accuracy)
    print('  Precision Score:', precision)
    print('     Recall Score:', recall)
    print('         F1 Score:', f1)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))


In [ ]:

RANDOM_SEED = 42
TEST_SIZE = 0.3

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(f"{path}/creditcard.csv")
data['Class'] = data['Class'].astype(int)

data = data.drop(['Time'], axis=1)
data['Amount'] = StandardScaler().fit_transform(data['Amount'].values.reshape(-1, 1))


In [ ]:

X = data.drop(columns=['Class']).values
Y = data['Class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=Y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
evaluation(y_test, y_pred, model_name="Random Forest (Original)")


In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED)
rf_model.fit(X_train, y_train.ravel())
y_pred = rf_model.predict(X_test)
evaluation(y_test, y_pred, model_name="Random Forest (Balanced)")


In [ ]:

x_train, x_test, y_train_k, y_test_k = train_test_split(
    X, Y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=Y)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

n_x_train = x_train[y_train_k == 0][:1000]
scores = []
for k in range(2, 5):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_SEED)
    kmeans.fit(n_x_train)
    scores.append(silhouette_score(n_x_train, kmeans.labels_))

optimal_k = np.argmax(scores) + 2
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=RANDOM_SEED)
kmeans.fit(n_x_train)
y_pred_test = kmeans.predict(x_test)

def align_labels(y_true, y_pred, n_clusters):
    labels = np.zeros_like(y_pred)
    for i in range(n_clusters):
        mask = (y_pred == i)
        if np.sum(mask) > 0:
            labels[mask] = np.bincount(y_true[mask]).argmax()
        else:
            labels[mask] = 0
    return labels

y_pred_aligned = align_labels(y_test_k, y_pred_test, optimal_k)
evaluation(y_test_k, y_pred_aligned, model_name="KMeans (Unsupervised)")
